<a href="https://colab.research.google.com/github/Success4A/pyTorch-wine-classification/blob/main/Wine_Variety_(PyTorch_Assignment).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Assignment: Wine Variety Classification

## Aim
Build a neural network that predicts which of three wine varieties a sample belongs to, using `sklearn.datasets.load_wine`.

This notebook adapts the lecture's binary-classification workflow (from my PyTorch Class) to the three classes in Scikit-learn's Wine dataset. Short comments explain what each code section and important line does.


## 1. Import the libraries

In [ ]:
# Import PyTorch, neural-network layers, optimizers, NumPy, and data-loading tools.
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Import tools for mini-batches, the wine dataset, splitting, scaling, and evaluation.
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Set the random seed so PyTorch's random operations are more reproducible.
torch.manual_seed(42)


## 2. Load the wine dataset

In [ ]:
# Load the wine dataset provided by Scikit-learn.
data = load_wine()

# Store the 13 measured wine features and their class labels separately.
X = data.data
y = data.target

# Display the dimensions of the data and the names of the three wine classes.
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Wine varieties:", data.target_names)


Features shape: (178, 13)
Target shape: (178,)
Wine varieties: ['class_0' 'class_1' 'class_2']


The dataset has 13 input features. The target labels are whole numbers representing the three wine classes: 0, 1, and 2.

## 3. Split the data into training and testing sets

In [ ]:
# Split the dataset into 80% training data and 20% test data.
# stratify=y helps keep the class proportions similar in both sets.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Show the number of rows and features in the training and test sets.
print("Training features:", X_tr.shape)
print("Testing features:", X_te.shape)


Training features: (142, 13)
Testing features: (36, 13)


## 4. Scale the features

Fit the scaler on the training data only, then use it to transform both training and test features.

In [ ]:
# Create a scaler that standardizes features using their training-set statistics.
sc = StandardScaler()

# Fit the scaler on the training features and transform them.
X_tr = sc.fit_transform(X_tr)

# Apply the same training-set scaling to the test features to avoid data leakage.
X_te = sc.transform(X_te)


## 5. Convert the data to PyTorch tensors

For this three-class task, the target labels use `torch.long` and are **not reshaped**. This is required by `nn.CrossEntropyLoss()`.

In [ ]:
# Convert input features to 32-bit floating-point tensors for the neural network.
X_tr = torch.tensor(np.asarray(X_tr), dtype=torch.float32)
X_te = torch.tensor(np.asarray(X_te), dtype=torch.float32)

# Convert class labels to integer tensors, as CrossEntropyLoss expects class indices.
# Keep labels one-dimensional; do not reshape them into a column.
y_tr = torch.tensor(np.asarray(y_tr), dtype=torch.long)
y_te = torch.tensor(np.asarray(y_te), dtype=torch.long)

# Check the tensor dimensions and data types.
print("X_tr:", X_tr.shape, X_tr.dtype)
print("y_tr:", y_tr.shape, y_tr.dtype)


X_tr: torch.Size([142, 13]) torch.float32
y_tr: torch.Size([142]) torch.int64


## 6. Create a DataLoader

In [ ]:
# Combine training features and labels into one dataset for batch training.
# The DataLoader supplies batches of 32 samples and shuffles them each epoch.
train_dl = DataLoader(
    TensorDataset(X_tr, y_tr),
    batch_size=32,
    shuffle=True
)


## 7. Define the neural network

The model follows the class structure, with hidden layers of 32 and 16 units. The final layer is `nn.Linear(16, 3)` because there are three wine classes. No sigmoid is needed for this model.

In [ ]:
# Define a neural network for the wine-classification task.
class Net(nn.Module):
    def __init__(self, n_in):
        # Initialize the parent PyTorch module.
        super().__init__()

        # Define the layers: input features, hidden layers, and three class scores.
        self.net = nn.Sequential(
            nn.Linear(n_in, 32),  # Map input features to 32 hidden units.
            nn.ReLU(),            # Add a non-linear activation.
            nn.Dropout(0.2),      # Randomly drop 20% of units during training.
            nn.Linear(32, 16),    # Map 32 hidden units to 16.
            nn.ReLU(),            # Apply another non-linear activation.
            nn.Dropout(0.2),      # Reduce over-reliance on particular units during training.
            nn.Linear(16, 3)      # Output one score for each of the three wine classes.
        )

    def forward(self, x):
        # Pass the input through the defined layers and return the class scores (logits).
        return self.net(x)


## 8. Set up the model, loss function, and optimizer

In [ ]:
# Use the GPU if available; otherwise, run the model on the CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create the model using the number of input features and move it to the selected device.
model = Net(X_tr.shape[1]).to(device)

# Use cross-entropy loss for classification into three classes.
criterion = nn.CrossEntropyLoss()

# Use Adam to update the model's trainable parameters.
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Display the device selected for training.
print("Using device:", device)


Using device: cuda


## 9. Train the model and print loss for every epoch

This uses the lecture's mini-batch training pattern. The assignment requires the loss to be printed for each epoch.

In [ ]:
# Train the model for 60 complete passes (epochs) through the training data.
epochs = 60

for epoch in range(epochs):
    # Set the model to training mode so dropout is active.
    model.train()

    # Start an accumulator for this epoch's batch losses.
    running = 0.0

    # Process the training set one mini-batch at a time.
    for xb, yb in train_dl:
        # Move the current batch of features and labels to the selected device.
        xb, yb = xb.to(device), yb.to(device)

        # Clear gradients left from the previous optimization step.
        optimizer.zero_grad()

        # Forward pass: calculate the three class scores for each sample.
        logits = model(xb)

        # Compare the class scores with the correct integer class labels.
        loss = criterion(logits, yb)

        # Backpropagate the loss to calculate gradients.
        loss.backward()

        # Update the model parameters using those gradients.
        optimizer.step()

        # Add this batch's loss to the epoch total.
        running += loss.item()

    # Calculate and print the mean batch loss for this epoch.
    epoch_loss = running / len(train_dl)
    print(f"Epoch {epoch + 1:3d}, Loss: {epoch_loss:.4f}")


Epoch   1, Loss: 1.0918
Epoch   2, Loss: 1.0754
Epoch   3, Loss: 1.0275
Epoch   4, Loss: 0.9955
Epoch   5, Loss: 0.9653
Epoch   6, Loss: 0.9163
Epoch   7, Loss: 0.9065
Epoch   8, Loss: 0.8633
Epoch   9, Loss: 0.8178
Epoch  10, Loss: 0.7964
Epoch  11, Loss: 0.7543
Epoch  12, Loss: 0.7167
Epoch  13, Loss: 0.6644
Epoch  14, Loss: 0.5769
Epoch  15, Loss: 0.5481
Epoch  16, Loss: 0.4810
Epoch  17, Loss: 0.5129
Epoch  18, Loss: 0.4271
Epoch  19, Loss: 0.3956
Epoch  20, Loss: 0.3524
Epoch  21, Loss: 0.3083
Epoch  22, Loss: 0.2654
Epoch  23, Loss: 0.2740
Epoch  24, Loss: 0.2455
Epoch  25, Loss: 0.2066
Epoch  26, Loss: 0.2080
Epoch  27, Loss: 0.1850
Epoch  28, Loss: 0.1476
Epoch  29, Loss: 0.1585
Epoch  30, Loss: 0.1548
Epoch  31, Loss: 0.1144
Epoch  32, Loss: 0.1104
Epoch  33, Loss: 0.1150
Epoch  34, Loss: 0.0892
Epoch  35, Loss: 0.1124
Epoch  36, Loss: 0.0815
Epoch  37, Loss: 0.1002
Epoch  38, Loss: 0.0913
Epoch  39, Loss: 0.0933
Epoch  40, Loss: 0.0690
Epoch  41, Loss: 0.0647
Epoch  42, Loss:

## 10. Evaluate the model and report test accuracy

For each test sample, `torch.argmax(logits, dim=1)` selects the class with the highest score.

In [ ]:
# Switch the model to evaluation mode; dropout is disabled for evaluation.
model.eval()

# Disable gradient calculations because the model is only being tested.
with torch.no_grad():
    # Calculate class scores for the held-out test features.
    logits = model(X_te.to(device))

    # Select the class with the highest score for each test sample.
    preds = torch.argmax(logits, dim=1).cpu()

# Compare predicted labels with the true test labels to calculate accuracy.
accuracy = accuracy_score(y_te.numpy(), preds.numpy())

# Print the test accuracy as a percentage.
print(f"Test Accuracy: {accuracy:.2%}")


Test Accuracy: 97.22%


## 11. Optional: confusion matrix and classification report

In [ ]:
# Display a table comparing actual classes with predicted classes.
print("Confusion matrix:")
print(confusion_matrix(y_te.numpy(), preds.numpy()))

# Show precision, recall, F1-score, and sample counts for each wine class.
print("\nClassification report:")
print(classification_report(
    y_te.numpy(),                 # Actual test labels.
    preds.numpy(),                 # Predicted test labels.
    target_names=data.target_names,  # Names of the three wine varieties.
    zero_division=0                # Show zero instead of a warning if a metric is undefined.
))


Confusion matrix:
[[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]

Classification report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



# **Environment and Hardware**

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


# README.md

This project uses a PyTorch neural network to classify samples from Scikit-learn's Wine dataset into three wine varieties. The workflow includes a train-test split, feature scaling, tensor conversion, mini-batch training with cross-entropy loss, and evaluation using test accuracy.